# 1. Scrapper básico utilizando Ollama
----

Notebook de experimentação com LLMs locais utilizando Ollama e Llama 3.2.

Objetivo:
- Executar modelos localmente
- Realizar Web Scraping
- Criar prompts
- Gerar resumos automáticos de websites
- Reproduzir o fluxo apresentado no curso do Ed Donner sem utilizar a OpenAI API


## Arquitetura

```text
Website
↓
Web Scraping
↓
Texto Limpo
↓
Messages
↓
Llama 3.2 (Ollama)
↓
Resumo
```


## Instalando bibliotecas necessárias
```
uv add openai requests beautifulsoup4 ipython python-dotenv
```

## Imports

Bibliotecas necessárias para comunicação com Ollama e scraping.

In [ ]:
from openai import OpenAI
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="anything"
)

print("Cliente criado")
print("Requests OK")
print("BeautifulSoup OK")

## Web Scraping

Função responsável por capturar e limpar o conteúdo do website.

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")

    title = soup.title.string if soup.title else "No title found"

    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()

        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""

    return (title + "\n\n" + text)[:2000]

## Prompt Engineering

- System Prompt = comportamento
- User Prompt = tarefa


In [ ]:
system_prompt = """
Você é um analista de sites.
Responda em Markdown.
"""

user_prompt_prefix = """
Aqui está o conteúdo de um site.
Forneça as principais noticías com uma breve opião em forma de sátira ou ironia.
"""

## Estrutura de Messages

Formata os dados no padrão esperado por APIs compatíveis com OpenAI.

In [ ]:
def messages_for(website):
    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt_prefix + website
        }
    ]

## Função de Resumo

Pipeline completo:

Website → Scraping → Prompt → Llama 3.2 → Resumo

In [ ]:
def summarize(url):
    website = fetch_website_contents(url)

    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=messages_for(website)
    )

    return response.choices[0].message.content

## Teste

In [ ]:
summary = summarize(
    "https://www.cnnbrasil.com.br/ia/"
)

display(
    Markdown(summary)
)